In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt
from time import strftime
from AIce.functions import trainloader,testloader,redim
from AIce.models import NNforNorms


In [11]:
inputlists=[['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm'],
            ['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos'],
            ['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath'],
            ['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE'],
            ['u_ERA5','v_ERA5','h_piomas','sic_CDR'],
            ['u_ERA5','v_ERA5','h_piomas','sic_CDR', 'windnorm'],
            ['u_ERA5','v_ERA5','h_piomas'],
            ['u_ERA5','v_ERA5']]

device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

In [ ]:
# ALL RIGHT LETS TRY TO TRAIN THAT THING
# Hyperparameters
lr=1e-3
n_epoch=20
print('Total Epoch: ', n_epoch)
print('Learning Rate: ', lr)


def RMSE(target,outputs):
    '''Calculates the RMSE between, the target and the model outputs'''
    return (torch.mean((target-outputs)**2))**.5


for listinput in inputlists:
    _,_,dataloader,_,_,_,labels=trainloader('../data/DRIFT_DATA_TRAIN.csv',256,listinput,target='buoynorm',)
    mlp256=NNforNorms(len(labels[1])).to(device)
    print('Total Epoch: ', n_epoch)
    print('Learning Rate: ', lr)

    rlosses={}
    losses=[]
    rloss=0
    rcount=0

    optimizer = optim.Adam(mlp256.parameters(),lr=lr) # optimizers
    for epoch in range(n_epoch):
        print('Starting Epoch ',epoch)
        curpercent=0
        rl=[]
        for i,data in enumerate(dataloader):

            # get the inputs and the target
            truth,inputs= data[0].to(device),data[1].to(device)

            # zero the gradients,
            optimizer.zero_grad()

            # Forward,
            out=mlp256(inputs)


            loss = RMSE(target=truth,outputs=out)
            # Backward
            loss.backward()
            losses.append(loss.item())
            # Optimize

            optimizer.step()

            rloss+= loss.item()
            rcount +=1
            #print where we at plus the loss
            percent= round(i/len(dataloader)*100)
            if percent != curpercent:
            #    print(percent, 'loss: ', rloss/rcount)
                curpercent=percent
                rl.append(rloss/rcount)
                rloss=0
                rcount=0
        rlosses[epoch]=rl
        print('Epoch avg loss: ', np.mean(rl))

    torch.save(mlp256.state_dict(), f'.//weights//NNforNorms_{len(labels[1])}inputs_{n_epoch}E_{lr}lr_{strftime("%d-%Hh_%Mm_%Ss")}.pt')

    with open(f'.//weights//NNforNorms_{len(labels[1])}inputs_{n_epoch}E_{lr}lr_{strftime("%d-%Hh_%Mm_%Ss")}.txt','w') as f:
        f.write(f'Model: mlp256 (on gpu)\n')
        f.write(f'Weights: NNforNorms_{len(labels[1])}inputs_{n_epoch}E_{lr}lr_{strftime("%d-%Hh_%Mm_%Ss")}.pt\n')
        f.write(f'Associated inputs: {listinput}')

    fig,ax =plt.subplots()
    epochavg=[]

    for i in (rlosses):
        x=np.arange(i*len(rlosses[0]),len(rlosses[0])+i*len(rlosses[0]))
        ax.plot(x,rlosses[i])#,label=f'Epoch {i+1}, lr={lr[i]:.1e}')
        avg=np.mean(rlosses[i])
        epochavg.append(avg)
    xx=[i*len(rlosses[0]) for i in rlosses]
    ax.plot(xx,epochavg,'.-k')
    plt.savefig(f'.//outputs//loss_for_NNforNorms_{len(labels[1])}inputs_{n_epoch}E_{lr}lr_{strftime("%d-%Hh_%Mm_%Ss")}.png')
    plt.close('all')

Total Epoch:  20
Learning Rate:  0.001
Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Total Epoch:  20
Learning Rate:  0.001
Starting Epoch  0
Epoch avg loss:  0.6585738274012948
Starting Epoch  1
Epoch avg loss:  0.60934738444103
Starting Epoch  2
Epoch avg loss:  0.5947399188791003
Starting Epoch  3
Epoch avg loss:  0.5813254132260988
Starting Epoch  4
Epoch avg loss:  0.5722915610278045
Starting Epoch  5
Epoch avg loss:  0.5656073076597281
Starting Epoch  6
Epoch avg loss:  0.5605674960488801
Starting Epoch  7
Epoch avg loss:  0.5562797498129881
Starting Epoch  8
Epoch avg loss:  0.5508915962229718
Starting Epoch  9
Epoch avg loss:  0.5469199565085736
Starting Epoch  10
Epoch avg loss:  0.5438904797113859
Starting Epoch  11
Epoch avg loss:  0.5382271384177627
Starting Epoch  12
Epoch avg loss:  0.5360572252447133